In [ ]:
!pip install mediapipe==0.10.14

import os
import cv2
import json
import numpy as np
import pandas as pd
import mediapipe as mp
from pathlib import Path
from tqdm import tqdm
from tensorflow.keras.utils import to_categorical

NUM_FRAMES = 30
INPUT_DIR = Path("/kaggle/input/datasets/vinphmngquc/asl-datasets")
VIDEO_DIR = INPUT_DIR / "videos"
OUTPUT_DIR = Path("/kaggle/working/dataset_landmarks")

TARGET_COUNT = 150
TOP_K = 10  

POSE_DIM = 33 * 3
HAND_DIM = 21 * 3

POSE_START = 0
POSE_END   = POSE_DIM
LH_START   = POSE_END
LH_END     = LH_START + HAND_DIM
RH_START   = LH_END
RH_END     = RH_START + HAND_DIM

for split in ['train', 'valid', 'test']:
    os.makedirs(OUTPUT_DIR / split, exist_ok=True)


def get_top_k_labels(train_csv, valid_csv, test_csv, k=10):
    def labels_with_videos(csv_path):
        df = pd.read_csv(csv_path)
        valid_labels = set()
        for label, group in df.groupby("Gloss"):
            if any((VIDEO_DIR / vid).exists() for vid in group["VideoID"]):
                valid_labels.add(label)
        return valid_labels

    print("Đang kiểm tra video tồn tại trên disk...")
    train_labels = labels_with_videos(train_csv)
    valid_labels = labels_with_videos(valid_csv)
    test_labels  = labels_with_videos(test_csv)

    common_labels = train_labels & valid_labels & test_labels
    print(f"  Train : {len(train_labels)} từ có video")
    print(f"  Valid : {len(valid_labels)} từ có video")
    print(f"  Test  : {len(test_labels)} từ có video")
    print(f"  Giao  : {len(common_labels)} từ có mặt ở cả 3 tập")

    if len(common_labels) == 0:
        raise ValueError("Không có từ nào chung giữa 3 tập! Kiểm tra lại đường dẫn video.")

    if len(common_labels) < k:
        print(f"  CẢNH BÁO: Chỉ có {len(common_labels)} từ chung, ít hơn TOP_K={k}.")
        print(f"  Sẽ lấy tất cả {len(common_labels)} từ.")
        k = len(common_labels)

    df_train = pd.read_csv(train_csv)
    df_train = df_train[df_train["Gloss"].isin(common_labels)]

    video_counts = []
    for label, group in df_train.groupby("Gloss"):
        count = sum(1 for vid in group["VideoID"] if (VIDEO_DIR / vid).exists())
        video_counts.append({"Gloss": label, "VideoCount": count})

    count_df = pd.DataFrame(video_counts).sort_values("VideoCount", ascending=False)
    top_labels = count_df.head(k)["Gloss"].tolist()

    print(f"\nTop {k} từ được chọn (có mặt ở cả 3 tập, nhiều video train nhất):")
    for _, row in count_df.head(k).iterrows():
        print(f"  {row['Gloss']:25s}: {row['VideoCount']} videos (train)")

    return top_labels


mp_holistic = mp.solutions.holistic

def extract_keypoints(results):
    pose = (
        np.array([[r.x, r.y, r.z]
                  for r in results.pose_landmarks.landmark], dtype=np.float32).flatten()
        if results.pose_landmarks else np.zeros(POSE_DIM, dtype=np.float32)
    )
    lh = (
        np.array([[r.x, r.y, r.z]
                  for r in results.left_hand_landmarks.landmark], dtype=np.float32).flatten()
        if results.left_hand_landmarks else np.zeros(HAND_DIM, dtype=np.float32)
    )
    rh = (
        np.array([[r.x, r.y, r.z]
                  for r in results.right_hand_landmarks.landmark], dtype=np.float32).flatten()
        if results.right_hand_landmarks else np.zeros(HAND_DIM, dtype=np.float32)
    )
    return np.concatenate([pose, lh, rh])


def normalize_body(seq):
    seq = seq.copy()
    T = seq.shape[0]
    pose = seq[:, POSE_START:POSE_END].reshape(T, 33, 3)
    center = (pose[:, 23] + pose[:, 24]) / 2.0
    scale = np.linalg.norm(pose[:, 11, :2] - pose[:, 12, :2], axis=1, keepdims=True) + 1e-6
    for t in range(T):
        for i in range(0, seq.shape[1], 3):
            seq[t, i:i+3] -= center[t]
            seq[t, i:i+3] /= scale[t]
    return seq


def add_velocity(seq):
    velocity = np.diff(seq, axis=0, prepend=seq[0:1])
    return np.concatenate([seq, velocity], axis=-1)


def uniform_sample(seq, num_frames):
    T = len(seq)
    if T == num_frames:
        return seq
    indices = np.linspace(0, T - 1, num_frames).astype(int)
    return seq[indices]


def video_to_sequence_optimized(path, holistic):
    cap = cv2.VideoCapture(str(path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        return None
        
    indices = set(np.linspace(0, total_frames - 1, NUM_FRAMES).astype(int))
    
    frames = []
    frame_idx = 0
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        if frame_idx in indices:
            frame = cv2.resize(frame, (640, int(640 * frame.shape[0] / frame.shape[1])))
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(img)
            frames.append(extract_keypoints(results))
            
        frame_idx += 1
        
    cap.release()
    
    if len(frames) == 0:
        return None
        
    seq = np.array(frames, dtype=np.float32)
    seq = uniform_sample(seq, NUM_FRAMES) 
    seq = normalize_body(seq)
    seq = add_velocity(seq)
    return seq


def aug_add_noise(seq, std=0.005):
    return seq + np.random.normal(0, std, seq.shape).astype(np.float32)

def aug_time_warp(seq, speed_factor=None):
    speed_factor = np.random.uniform(0.7, 1.3) if speed_factor is None else speed_factor
    T, D = seq.shape
    new_len = max(2, int(T * speed_factor))
    old_t = np.arange(T, dtype=np.float32)
    mid_t = np.linspace(0, T - 1, new_len, dtype=np.float32)
    fin_t = np.linspace(0, new_len - 1, T, dtype=np.float32)
    warped = np.empty((new_len, D), dtype=np.float32)
    for d in range(D): warped[:, d] = np.interp(mid_t, old_t, seq[:, d])
    result = np.empty((T, D), dtype=np.float32)
    w_old = np.arange(new_len, dtype=np.float32)
    for d in range(D): result[:, d] = np.interp(fin_t, w_old, warped[:, d])
    return result

def aug_scale(seq, scale=None):
    scale = np.random.uniform(0.9, 1.1) if scale is None else scale
    result = seq.copy()
    T = seq.shape[0]
    pose = result[:, POSE_START:POSE_END].reshape(T, 33, 3)
    center = (pose[:, 23] + pose[:, 24]) / 2
    for t in range(T):
        for i in range(0, result.shape[1], 3):
            result[t, i:i+3] = center[t] + scale * (result[t, i:i+3] - center[t])
    return result

def aug_temporal_shift(seq, max_shift=3):
    shift = np.random.randint(-max_shift, max_shift + 1)
    if shift == 0: return seq
    shifted = np.roll(seq, shift, axis=0)
    if shift > 0: shifted[:shift] = seq[0]
    else: shifted[shift:] = seq[-1]
    return shifted

def aug_frame_dropout(seq, drop_rate=0.05):
    result = seq.copy()
    T = len(seq)
    n_drop = max(1, int(T * drop_rate))
    for idx in np.random.choice(T, n_drop, replace=False):
        result[idx] = result[max(0, idx - 1)]
    return result

def aug_rotation_2d(seq, max_angle_deg=10.0):
    theta = np.radians(np.random.uniform(-max_angle_deg, max_angle_deg))
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    result = seq.copy()
    T = seq.shape[0]
    pose = result[:, POSE_START:POSE_END].reshape(T, 33, 3)
    center = (pose[:, 23] + pose[:, 24]) / 2
    for t in range(T):
        for i in range(0, result.shape[1], 3):
            x, y = result[t, i] - center[t, 0], result[t, i+1] - center[t, 1]
            result[t, i]   = cos_t * x - sin_t * y + center[t, 0]
            result[t, i+1] = sin_t * x + cos_t * y + center[t, 1]
    return result

AUGMENTATION_FUNCS = [aug_add_noise, aug_time_warp, aug_scale,
                       aug_temporal_shift, aug_frame_dropout, aug_rotation_2d]

def _apply_random_augs(seq):
    n_augs = np.random.randint(1, 4)
    chosen = np.random.choice(len(AUGMENTATION_FUNCS), n_augs, replace=False)
    aug = seq.copy()
    for idx in chosen: aug = AUGMENTATION_FUNCS[idx](aug)
    return aug

def augment_to_target(base_sequences, target_count):
    n_base = len(base_sequences)
    if n_base == 0: return []
    samples = list(base_sequences)
    needed = target_count - n_base
    if needed <= 0: return samples[:target_count]
    base_quota, remainder = needed // n_base, needed % n_base
    aug_pool = []
    for i, base_seq in enumerate(base_sequences):
        quota = base_quota + (1 if i < remainder else 0)
        for _ in range(quota): aug_pool.append(_apply_random_augs(base_seq))
    samples.extend(aug_pool)
    return samples[:target_count]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.25 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 require

2026-05-08 17:27:14.789245: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778261235.056409      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778261235.135703      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778261235.749157      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778261235.749219      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778261235.749222      16 computation_placer.cc:177] computation placer alr

In [ ]:
import concurrent.futures


def process_single_video(vid_path, output_dir, label, idx, model_complexity=0):
    with mp_holistic.Holistic(
        model_complexity=model_complexity, 
        min_detection_confidence=0.5, 
        min_tracking_confidence=0.5
    ) as holistic:
        seq = video_to_sequence_optimized(vid_path, holistic)
        if seq is not None:
            return seq
    return None

def process_dataset_parallel(csv_path, split, allowed_labels):
    df = pd.read_csv(csv_path)
    df = df[df["Gloss"].isin(allowed_labels)]
    print(f"[{split}] {df['Gloss'].nunique()} từ | {len(df)} video rows")
    
    for label, group in tqdm(df.groupby("Gloss"), desc=split):
        class_dir = OUTPUT_DIR / split
        os.makedirs(class_dir, exist_ok=True)
        
        video_paths = [VIDEO_DIR / vid for vid in group["VideoID"] if (VIDEO_DIR / vid).exists()]
        
        with concurrent.futures.ProcessPoolExecutor() as executor:
            results = list(executor.map(
                process_single_video, 
                video_paths, 
                [class_dir] * len(video_paths), 
                [label] * len(video_paths), 
                range(len(video_paths))
            ))
            
        seqs = [r for r in results if r is not None]

        if split == "train" and len(seqs) > 0:
            seqs = augment_to_target(seqs, TARGET_COUNT)
        
        if len(seqs) > 0:
            stacked_seqs = np.array(seqs)
            np.save(class_dir / f"{label}.npy", stacked_seqs)


top_labels = get_top_k_labels(
    INPUT_DIR / "train.csv",
    INPUT_DIR / "valid.csv",
    INPUT_DIR / "test.csv",
    k=TOP_K
)
print(f"\nSẽ xử lý: {top_labels}")




Đang kiểm tra video tồn tại trên disk...
  Train : 126 từ có video
  Valid : 126 từ có video
  Test  : 126 từ có video
  Giao  : 126 từ có mặt ở cả 3 tập

Top 10 từ được chọn (có mặt ở cả 3 tập, nhiều video train nhất):
  NIGHT                    : 169 videos (train)
  READ                     : 149 videos (train)
  FINISH                   : 147 videos (train)
  FRIEND                   : 143 videos (train)
  WHERE                    : 138 videos (train)
  WATER                    : 138 videos (train)
  MANY                     : 133 videos (train)
  WRITE                    : 132 videos (train)
  START                    : 114 videos (train)
  YOU                      : 113 videos (train)

Sẽ xử lý: ['NIGHT', 'READ', 'FINISH', 'FRIEND', 'WHERE', 'WATER', 'MANY', 'WRITE', 'START', 'YOU']


In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

def resplit_csvs(train_csv, valid_csv, test_csv, top_labels,
                 train_ratio=0.70, valid_ratio=0.15, test_ratio=0.15):
    """
    Gộp toàn bộ dữ liệu từ 3 CSV gốc (chỉ giữ top_labels),
    rồi chia lại theo tỉ lệ 70/15/15 theo từng class (stratified).
    Trả về 3 DataFrame mới: df_train, df_valid, df_test
    """
    assert abs(train_ratio + valid_ratio + test_ratio - 1.0) < 1e-6, \
        "Tổng tỉ lệ phải bằng 1.0"

    # Gộp 3 CSV, chỉ giữ các từ trong top_labels
    df_all = pd.concat([
        pd.read_csv(train_csv),
        pd.read_csv(valid_csv),
        pd.read_csv(test_csv),
    ], ignore_index=True)
    df_all = df_all[df_all["Gloss"].isin(top_labels)].copy()

    # Chỉ giữ những video thực sự tồn tại trên disk
    df_all = df_all[df_all["VideoID"].apply(lambda v: (VIDEO_DIR / v).exists())].copy()
    df_all = df_all.drop_duplicates(subset=["VideoID"]).reset_index(drop=True)

    print(f"Tổng video hợp lệ sau khi gộp: {len(df_all)}")
    print(f"Phân phối theo class:\n{df_all['Gloss'].value_counts().to_string()}\n")

    # Chia stratified:
    # Bước 1 - tách test (15%)
    # Bước 2 - tách valid từ phần còn lại: 15% / 85% ≈ 17.65%
    test_size  = test_ratio
    valid_size = valid_ratio / (1.0 - test_ratio)

    df_trainval, df_test = train_test_split(
        df_all, test_size=test_size,
        stratify=df_all["Gloss"], random_state=RANDOM_STATE
    )
    df_train, df_valid = train_test_split(
        df_trainval, test_size=valid_size,
        stratify=df_trainval["Gloss"], random_state=RANDOM_STATE
    )

    total = len(df_all)
    print("Kết quả chia lại:")
    print(f"  Train : {len(df_train):4d} videos ({len(df_train)/total*100:.1f}%)")
    print(f"  Valid : {len(df_valid):4d} videos ({len(df_valid)/total*100:.1f}%)")
    print(f"  Test  : {len(df_test):4d} videos ({len(df_test)/total*100:.1f}%)")

    return df_train.reset_index(drop=True), \
           df_valid.reset_index(drop=True), \
           df_test.reset_index(drop=True)


df_train_new, df_valid_new, df_test_new = resplit_csvs(
    INPUT_DIR / "train.csv",
    INPUT_DIR / "valid.csv",
    INPUT_DIR / "test.csv",
    top_labels
)

# (Tuỳ chọn) Lưu CSV mới ra disk để kiểm tra
RESPLIT_DIR = OUTPUT_DIR / "resplit_csv"
os.makedirs(RESPLIT_DIR, exist_ok=True)
df_train_new.to_csv(RESPLIT_DIR / "train.csv", index=False)
df_valid_new.to_csv(RESPLIT_DIR / "valid.csv", index=False)
df_test_new.to_csv(RESPLIT_DIR  / "test.csv",  index=False)
print(f"\nĐã lưu CSV chia lại vào: {RESPLIT_DIR}")


Tổng video hợp lệ sau khi gộp: 1722
Phân phối theo class:
Gloss
NIGHT     212
READ      186
FINISH    184
FRIEND    179
WATER     173
WHERE     173
MANY      167
WRITE     165
START     142
YOU       141

Kết quả chia lại:
  Train : 1204 videos (69.9%)
  Valid :  259 videos (15.0%)
  Test  :  259 videos (15.0%)

Đã lưu CSV chia lại vào: /kaggle/working/dataset_landmarks/resplit_csv


In [ ]:
def process_dataset_from_df(df, split, allowed_labels):
    """Giống process_dataset_parallel nhưng nhận DataFrame thay vì đường dẫn CSV."""
    df = df[df["Gloss"].isin(allowed_labels)]
    print(f"[{split}] {df['Gloss'].nunique()} từ | {len(df)} video rows")

    for label, group in tqdm(df.groupby("Gloss"), desc=split):
        class_dir = OUTPUT_DIR / split
        os.makedirs(class_dir, exist_ok=True)

        video_paths = [VIDEO_DIR / vid for vid in group["VideoID"]
                       if (VIDEO_DIR / vid).exists()]

        with concurrent.futures.ProcessPoolExecutor() as executor:
            results = list(executor.map(
                process_single_video,
                video_paths,
                [class_dir] * len(video_paths),
                [label]     * len(video_paths),
                range(len(video_paths))
            ))

        seqs = [r for r in results if r is not None]

        if split == "train" and len(seqs) > 0:
            seqs = augment_to_target(seqs, TARGET_COUNT)

        if len(seqs) > 0:
            stacked_seqs = np.array(seqs)
            np.save(class_dir / f"{label}.npy", stacked_seqs)


process_dataset_from_df(df_train_new, "train", top_labels)
process_dataset_from_df(df_valid_new, "valid", top_labels)
process_dataset_from_df(df_test_new,  "test",  top_labels)


[train] 10 từ | 1204 video rows


train:   0%|          | 0/10 [00:00<?, ?it/s]

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778261279.741997     108 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778261279.770531     106 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778261279.774066     110 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778261279.778093     103 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778261279.784484     106 inference_

[valid] 10 từ | 259 video rows


valid:   0%|          | 0/10 [00:00<?, ?it/s]INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778262950.158105   13612 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778262950.176051   13613 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778262950.188241   13612 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778262950.191777   13612 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W00

[test] 10 từ | 259 video rows


test:   0%|          | 0/10 [00:00<?, ?it/s]INFO: INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778263314.119561   16712 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778263314.141432   16718 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778263314.150029   16721 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778263314.163441   16709 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W000

In [5]:
class_names = sorted([f.replace('.npy', '') for f in os.listdir(OUTPUT_DIR / "train") if f.endswith('.npy')])
label_map = {c: i for i, c in enumerate(class_names)}

with open(OUTPUT_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f)

print(f"\nLabel map ({len(label_map)} classes):")
for name, idx in label_map.items():
    print(f"  {idx}: {name}")


def load(split):
    X_list = []
    y_list = []
    
    for c, idx in label_map.items():
        file_path = OUTPUT_DIR / split / f"{c}.npy"
        
        if not file_path.exists():
            print(f"  CẢNH BÁO: Không tìm thấy file {file_path}")
            continue
            
        class_data = np.load(file_path)
        
        if len(class_data) == 0:
            print(f"  CẢNH BÁO: File {file_path} trống")
            continue
            
        X_list.append(class_data)
        y_list.extend([idx] * len(class_data))
        

    if len(X_list) > 0:
        X = np.concatenate(X_list, axis=0)
    else:
        X = np.array([])
        
    y = np.array(y_list)
    
    return X, y


X_train, y_train = load("train")
X_val,   y_val   = load("valid")
X_test,  y_test  = load("test")

print(f"\nX_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}   | y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}  | y_test : {y_test.shape}")

y_train = to_categorical(y_train)
y_val   = to_categorical(y_val)
y_test  = to_categorical(y_test)


# =========================================================
# SAVE FINAL
# =========================================================

FINAL = OUTPUT_DIR / "final"
os.makedirs(FINAL, exist_ok=True)

np.save(FINAL / "X_train.npy", X_train)
np.save(FINAL / "y_train.npy", y_train)
np.save(FINAL / "X_val.npy",   X_val)
np.save(FINAL / "y_val.npy",   y_val)
np.save(FINAL / "X_test.npy",  X_test)
np.save(FINAL / "y_test.npy",  y_test)

print("\nDONE")


Label map (10 classes):
  0: FINISH
  1: FRIEND
  2: MANY
  3: NIGHT
  4: READ
  5: START
  6: WATER
  7: WHERE
  8: WRITE
  9: YOU

X_train: (1500, 30, 450) | y_train: (1500,)
X_val  : (259, 30, 450)   | y_val  : (259,)
X_test : (259, 30, 450)  | y_test : (259,)

DONE
